# 0. Data Preparation

**Dataset:** Genome-scale Perturb-seq — Primary Human CD4+ T Cells (Zhu, Dann et al. 2025)  
**Reference:** https://virtualcellmodels.cziscience.com/dataset/genome-scale-tcell-perturb-seq

This notebook handles all one-time data preparation tasks that do not depend on which condition is being analysed downstream.

**NTC extraction:** Each raw per-donor h5ad file (>100 GB) contains all perturbed and non-targeting control (NTC) cells. We filter to NTC cells only and save a much smaller file per donor per condition. These smaller files are what Notebook 1 uses to compute the stimulation vector.

## Contents
1. Configuration
2. NTC extraction

### 1. Configuration

In [1]:
import os
print(f'Working directory: {os.getcwd()}')

Working directory: /mnt/R0/Projects/POIAZ/Ilaria/Scripts


In [2]:
import numpy as np
import pandas as pd
import anndata

# Data paths
DATA_DIR = '../Data'
DESTATS_PATH = '../Data/GWCD4i.DE_stats.h5ad'

# Output folder for shared reference files (condition-agnostic outputs)
REF_DIR = '../Results/ref'
os.makedirs(REF_DIR, exist_ok=True)

# NTC extraction settings
# Run one donor+condition at a time — files are >100 GB and cannot be looped
DONOR = 'D1'  # 'D1', 'D2', 'D3', 'D4'
CONDITION = 'Rest'  # 'Rest', 'Stim8hr', 'Stim48hr'

P_THRESHOLD = 0.05

print(f'Data dir: {DATA_DIR}')
print(f'Ref dir: {REF_DIR}')
print(f'NTC target: {DONOR}_{CONDITION}')

Data dir: ../Data
Ref dir: ../Results/ref
NTC target: D1_Rest


### 2. NTC Extraction

Filter the raw assigned-guide h5ad to cells with `guide_type == 'non-targeting'` and save.

**Important:** Run one donor+condition at a time by changing `DONOR` and `CONDITION` in section 1 and re-running this cell. The raw files are >100 GB and loading multiple simultaneously will exhaust RAM.

**Files already extracted (update this list as you go):**
- D1_Rest.NTC.h5ad
- D1_Stim8hr.NTC.h5ad
- D1_Stim48hr.NTC.h5ad
- D2_Rest.NTC.h5ad
- D2_Stim8hr.NTC.h5ad
- D2_Stim48hr.NTC.h5ad
- D3_Rest.NTC.h5ad
- D3_Stim8hr.NTC.h5ad
- D3_Stim48hr.NTC.h5ad
- D4_Rest.NTC.h5ad
- D4_Stim8hr.NTC.h5ad
- D4_Stim48hr.NTC.h5ad

In [3]:
input_file = os.path.join(DATA_DIR, f'{DONOR}_{CONDITION}.assigned_guide.h5ad')
output_file = os.path.join(DATA_DIR, f'{DONOR}_{CONDITION}.NTC.h5ad')

if os.path.exists(output_file):
    print(f'Output already exists: {output_file}')
    print('Delete it manually if you want to re-extract.')
else:
    print(f'Loading: {input_file}')
    adata = anndata.read_h5ad(input_file)
    print(f'Loaded: {adata.shape}')

    if 'guide_type' not in adata.obs.columns:
        raise KeyError('guide_type column not found in adata.obs')

    ntc = adata[adata.obs['guide_type'] == 'non-targeting'].copy()
    print(f'NTC cells: {ntc.n_obs:,} out of {adata.n_obs:,} total')

    ntc.write_h5ad(output_file)
    print(f'Saved: {output_file}')

    del adata, ntc

Output already exists: ../Data/D1_Rest.NTC.h5ad
Delete it manually if you want to re-extract.
